# Echelon Chess Engine - Colab Training

This notebook trains the Echelon chess engine using AlphaZero-style self-play.

**Before running:**
1. Go to Runtime > Change runtime type > GPU (T4)
2. Upload or clone the echelon repository

In [ ]:
# Clone repository (or upload files)
!git clone https://github.com/falloficarus22/echelon.git
%cd echelon

In [ ]:
# Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Test the model
!python model.py

In [ ]:
# Test move encoder
!python move_encoder.py

In [ ]:
# Test MCTS (short test)
!python mcts.py

## Training

**Recommended settings for T4 (15GB VRAM):**
- `--num_filters 256` - Full ResNet-20 with 256 filters
- `--batch_size 256` - Fits well in T4 memory
- `--mcts_sims 200` - Balance between quality and speed
- `--games_per_iter 25` - Reasonable for Colab time limits

**For Kaggle P100 (16GB VRAM):**
- Same settings work well

In [ ]:
# Start training (short run to test)
!python train.py \
    --iterations 5 \
    --games_per_iter 10 \
    --batches_per_iter 100 \
    --batch_size 256 \
    --mcts_sims 100 \
    --num_filters 128 \
    --num_blocks 5

In [ ]:
# Full training run (~3-4 hours on T4)
!python train.py \
    --iterations 25 \
    --games_per_iter 20 \
    --batches_per_iter 100 \
    --batch_size 256 \
    --mcts_sims 100 \
    --num_filters 128 \
    --num_blocks 5

In [ ]:
# Resume training from checkpoint
!python train.py \
    --iterations 50 \
    --games_per_iter 25 \
    --resume checkpoints/latest.pt

## Save Results to Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Copy checkpoints to Drive
import shutil
import os

drive_path = '/content/drive/MyDrive/echelon_checkpoints'
os.makedirs(drive_path, exist_ok=True)

# Copy all checkpoints
!cp -r checkpoints/* {drive_path}/
print(f"Saved checkpoints to {drive_path}")

## Test Trained Model

In [ ]:
# Watch the trained engine play
from model import EchelonNet
from engine import BoardState
from mcts import MCTS
import torch

# Load best model
model = EchelonNet(in_channels=13, num_res_blocks=5, num_filters=128)
checkpoint = torch.load('checkpoints/best.pt', map_location='cuda')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Loaded model from iteration {checkpoint['iteration']}")

In [ ]:
# Play a quick game
board = BoardState()
board.parse_fen("rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1")

mcts = MCTS(model, num_simulations=400, temperature=0.1)

for i in range(10):
    legal_moves = board.generate_legal_moves(board.side)
    if not legal_moves:
        print("Game over!")
        break
    
    best_move, probs = mcts.search(board)
    decoded = board.decode_move(best_move)
    from_sq, to_sq = decoded['from'], decoded['to']
    move_str = f"{chr(ord('a') + from_sq % 8)}{from_sq // 8 + 1}"
    move_str += f"{chr(ord('a') + to_sq % 8)}{to_sq // 8 + 1}"
    
    side = "White" if board.side == 0 else "Black"
    print(f"{side}: {move_str}")
    
    board.make_move(best_move)